<a href="https://colab.research.google.com/github/ArielSmoliar/llm-finetuning-lab/blob/main/01_stack_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT_DIR = Path("/content/drive/MyDrive/llm-finetuning-lab")
for folder in ["data", "notebooks", "outputs", "experiments"]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print(f"Project folder: {PROJECT_DIR}")


Mounted at /content/drive
Project folder: /content/drive/MyDrive/llm-finetuning-lab


In [4]:
import torch

assert torch.cuda.is_available(), "No GPU found. Change the Colab runtime to GPU."
gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("PyTorch:", torch.__version__)
print("GPU:", gpu_name)
print(f"GPU memory: {gpu_memory:.1f} GB")
print("BF16 supported:", torch.cuda.is_bf16_supported())

PyTorch: 2.11.0+cu128
GPU: Tesla T4
GPU memory: 14.6 GB
BF16 supported: True


In [8]:
import torch

assert torch.cuda.is_available(), "No GPU found. Change the Colab runtime to GPU."
gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("PyTorch:", torch.__version__)
print("GPU:", gpu_name)
print(f"GPU memory: {gpu_memory:.1f} GB")

print("Native BF16 supported:",
      torch.cuda.is_bf16_supported(including_emulation=False))





PyTorch: 2.11.0+cu128
GPU: Tesla T4
GPU memory: 14.6 GB
Native BF16 supported: False


In [9]:
%pip install -q -U \
  "transformers>=5.10.1" \
  datasets accelerate evaluate bitsandbytes \
  trl "peft>=0.19.0" huggingface_hub \
  sentencepiece tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 30.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.20.0 requires tensorboard~=2.20.0, but you have tensorboard 2.21.0 which is incompatible.
google

In [10]:
%pip install "tensorboard~=2.20.0" "protobuf>=5.29.1,<6"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 33.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.36.1
    Uninstalling protobuf-7.36.1:
      Successfully uninstalled protobuf-7.36.1
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.21.0
    Uninstalling tensorboard-2.21.0:
      Successfully uninstalled tensorboard-2.21.0


In [1]:
import torch
import transformers
import datasets
import peft
import trl
import accelerate
import tensorboard
import google.protobuf

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("TensorBoard:", tensorboard.__version__)
print("Protobuf:", google.protobuf.__version__)

assert torch.cuda.is_available(), "No GPU found. Select a GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))
print(
    "Native BF16 supported:",
    torch.cuda.is_bf16_supported(including_emulation=False),
)

print("\nImports and GPU check passed.")

PyTorch: 2.11.0+cu128
Transformers: 5.17.0
Datasets: 5.0.1
PEFT: 0.20.0
TRL: 1.13.0
Accelerate: 1.15.0
TensorBoard: 2.20.0
Protobuf: 5.29.6
GPU: Tesla T4
Native BF16 supported: False

Imports and GPU check passed.


In [2]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
profile = whoami()
print("Authenticated as:", profile["name"])


Authenticated as: arielsmoliar


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype="auto",
)

messages = [{
    "role": "user",
    "content": "Classify this feedback as positive, neutral, or negative: I cannot verify my account."
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=40, do_sample=False)

new_tokens = output[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

**Negative**

This feedback indicates a significant problem – the user cannot access their account, which is a frustrating and potentially damaging experience. It’s clearly a negative situation.


In [4]:
from datetime import datetime, timezone
import json

CONFIG = {
    "experiment_id": "exp-001-smoke-test",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "dataset_version": "feedback-v1",
    "seed": 42,
    "max_length": 512,
    "lora_rank": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "epochs": 1,
}

config_path = PROJECT_DIR / "experiments" / f'{CONFIG["experiment_id"]}.json'
config_path.write_text(json.dumps(CONFIG, indent=2))
print(config_path)
print(json.dumps(CONFIG, indent=2))

NameError: name 'PROJECT_DIR' is not defined

In [5]:
from google.colab import drive
from pathlib import Path
import json

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/llm-finetuning-lab")
(PROJECT_DIR / "experiments").mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/llm-finetuning-lab


In [6]:
from datetime import datetime, timezone
import json

CONFIG = {
    "experiment_id": "exp-001-smoke-test",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "dataset_version": "feedback-v1",
    "seed": 42,
    "max_length": 512,
    "lora_rank": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "epochs": 1,
}

config_path = PROJECT_DIR / "experiments" / f'{CONFIG["experiment_id"]}.json'
config_path.write_text(json.dumps(CONFIG, indent=2))
print(config_path)
print(json.dumps(CONFIG, indent=2))

/content/drive/MyDrive/llm-finetuning-lab/experiments/exp-001-smoke-test.json
{
  "experiment_id": "exp-001-smoke-test",
  "created_at": "2026-09-11T16:56:17.726067+00:00",
  "model_id": "google/gemma-3-1b-it",
  "dataset_version": "feedback-v1",
  "seed": 42,
  "max_length": 512,
  "lora_rank": 16,
  "lora_alpha": 16,
  "learning_rate": 0.0002,
  "epochs": 1
}
